# MCA/M.Sc. (CS) Semester-II Mid-semester Examination 2024-25
## CS322P: Deep Learning

**Time:** 1 hour                                
**Marks:** 20

---

Consider the following neural network:

z₁ = W₁ x⁽ⁱ⁾ + b₁  
a₁ = tanh(z₁)  
z₂ = W₂ a₁ + b₂  
ŷ⁽ⁱ⁾ = softmax(z₂)

The loss function is given by:

L⁽ⁱ⁾ = - Σ (yⱼ⁽ⁱ⁾ log(ŷⱼ⁽ⁱ⁾))  for j = 1 to C  
J = (-1/m) Σ L⁽ⁱ⁾  for i = 1 to m

Where:
- x⁽ⁱ⁾ is an input example of shape (Dx = 200) × 1  
- y⁽ⁱ⁾ is a one-hot encoded label vector of shape (C = 10) × 1  
- m = 100 is the number of examples  
- Hidden layer has Da₁ = 20 neurons  

---

## Based on the above, answer the following:

(a) Generate 100 random synthetic samples  
    X ∈ R^(100 × 200)

(b) Generate 100 random class labels (0 to 9) and convert them into one-hot vectors  
    Y ∈ R^(100 × 200)

(c) Implement the above neural network from scratch in Python  
    (Do not use any ML libraries)

(d) Use randomly initialized weights and biases for one forward pass

(e) Compute the average cross-entropy loss

(f) Calculate and display the following metrics:

    - Accuracy  
    - Precision  
    - Recall  
    - F1-score

### (a) Synthetic Data Generation
We generate a dataset $X$ consisting of $m=100$ samples, where each sample $x^{(i)}$ is a vector of $D_x=200$ features. The samples are drawn from a standard normal distribution.

$X \in \mathbb{R}^{m \times D_x}$ where $m=100$ and $D_x=200$.

In [5]:
import numpy as np

# (a) Generate 100 random synthetic samples
np.random.seed(42) # For reproducibility
m = 100
Dx = 200

X = np.random.randn(m, Dx)
print(f"Shape of X: {X.shape}")
print(X)

Shape of X: (100, 200)
[[ 0.49671415 -0.1382643   0.64768854 ...  0.15372511  0.05820872
  -1.1429703 ]
 [ 0.35778736  0.56078453  1.08305124 ...  1.35387237 -0.11453985
   1.23781631]
 [-1.59442766 -0.59937502  0.0052437  ... -0.97876372 -0.44429326
   0.37730049]
 ...
 [ 0.72754737 -0.14897166 -0.78682763 ...  0.04420617  0.83211022
   0.54696847]
 [-0.26448035 -0.5438016  -0.15994215 ... -0.28670632  0.67016726
  -0.03139594]
 [ 1.90877627  0.69382541  0.62881383 ...  0.37835397  1.71352973
  -1.6199198 ]]


In [6]:
# Testing if mean = 0 ?
sum =0
for i in range(0,100):
    for j in range(0,200):
        sum += X[i][j]

mean = sum / 20000
print("Mean = ",mean)

Mean =  0.0056990348482305235


In [7]:
# Distributions
normal_dist = np.random.randn(3,2) # Uniform Distribution
print(normal_dist)

[[ 0.34828625  0.28332359]
 [-0.93651985  0.57958422]
 [-1.49008268 -0.65418433]]


### Understanding the Tools

#### 1. `np.random.seed(42)`
Random numbers in computing are actually **pseudo-random**. By setting a `seed`, we initialize the random number generator.
- **Why use it?** If you run the code again, or if I run it on my machine, we will get the exact same results. This is essential for reproducibility in Deep Learning experiments.

#### 2. Standard Normal Distribution
This is a specific probability distribution with two main properties:
- **Mean (Average) = 0**
- **Standard Deviation = 1**

#### 3. `np.random.randn(m, Dx)`
- The `n` in `randn` stands for **Normal**.
- This function draws samples from the Standard Normal Distribution.
- Roughly 68% of the numbers generated will fall between -1 and 1, and 95% will fall between -2 and 2.

In [ ]:
!pip install seaborn
import matplotlib.pyplot as plt
import seaborn as sns

# Let's visualize a large sample to see the 'Bell Curve'
large_sample = np.random.randn(10000)

plt.figure(figsize=(10, 5))
sns.histplot(large_sample, kde=True, color='skyblue')
plt.title("Visualization of Standard Normal Distribution (randn)")
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.axvline(x=0, color='red', linestyle='--', label='Mean (0)')
plt.legend()
plt.show()

print(f"Mean of sample: {np.mean(large_sample):.4f}")
print(f"Std dev of sample: {np.std(large_sample):.4f}")

^C


ModuleNotFoundError: No module named 'seaborn'

   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   -----------------

### (b) Ground Truth Generation
We generate $m=100$ random class labels from 0 to 9 and convert them into one-hot encoded vectors $Y \in \mathbb{R}^{100 \times 10}$.

A one-hot vector for class $j$ has a 1 at index $j$ and 0 elsewhere.

In [ ]:
import numpy as np

# (b) Generate 100 random class labels (0 to 9) and convert to one-hot
C = 10
y_true_labels = np.random.randint(0, C, size=m)

def to_one_hot(labels, num_classes):
    one_hot = np.zeros((labels.size, num_classes))
    one_hot[np.arange(labels.size), labels] = 1
    return medical_one_hot if 'medical_one_hot' in locals() else one_hot

Y = to_one_hot(y_true_labels, C)
print(f"Shape of Y: {Y.shape}")

Shape of Y: (100, 10)


### (c) & (d) Model Implementation and Forward Pass
The network consists of a hidden layer with a `tanh` activation and an output layer with a `softmax` activation.

**Forward equations:**
1. $z_1 = X W_1 + b_1$
2. $a_1 = \tanh(z_1)$
3. $z_2 = a_1 W_2 + b_2$
4. $\hat{y} = \text{softmax}(z_2)$

**Dimensions:**
- $X: (100 \times 200)$
- $W_1: (200 \times 20)$
- $b_1: (1 \times 20)$
- $W_2: (20 \times 10)$
- $b_2: (1 \times 10)$

In [ ]:
def tanh(z):
    return np.tanh(z)

def softmax(z):
    # Subtracting max for numerical stability
    z = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

# (c) & (d) Network implementation and forward pass
Da1 = 20  # Hidden layer neurons

# Random initialization
W1 = np.random.randn(Dx, Da1)
b1 = np.random.randn(1, Da1)
W2 = np.random.randn(Da1, C)
b2 = np.random.randn(1, C)

# Forward Pass
# z1 = X W1 + b1 (using matrix form for the batch m=100)
z1 = np.dot(X, W1) + b1
a1 = tanh(z1)
z2 = np.dot(a1, W2) + b2
y_hat = softmax(z2)

print(f"Shape of y_hat: {y_hat.shape}")

Shape of y_hat: (100, 10)


### (e) Average Cross-Entropy Loss
The loss for a single example $i$ is:
$L^{(i)} = - \sum_{j=1}^{C} y_j^{(i)} \log(\hat{y}_j^{(i)})$

The total cost $J$ is the average loss over $m$ examples:
$J = \frac{1}{m} \sum_{i=1}^{m} L^{(i)}$

In [ ]:
# (e) Compute average cross-entropy loss
# L = -sum(Y * log(y_hat))
# J = (1/m) * sum(L)
epsilon = 1e-12 # to avoid log(0)
loss = -np.sum(Y * np.log(y_hat + epsilon)) / m
print(f"Average Cross-Entropy Loss: {loss:.4f}")

Average Cross-Entropy Loss: 6.9023


### (f) Evaluation Metrics
We evaluate the model using:
- **Accuracy**: Fraction of correct predictions.
- **Precision**: $\frac{TP}{TP + FP}$
- **Recall**: $\frac{TP}{TP + FN}$
- **F1-score**: $2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$

Metrics are calculated per class and then averaged (Macro-averaging).

In [ ]:
# (f) Calculate Metrics
predictions = np.argmax(y_hat, axis=1)

def compute_metrics(y_true, y_pred, num_classes):
    # Accuracy
    accuracy = np.mean(y_true == y_pred)

    # Initialize metrics
    precision_list = []
    recall_list = []
    f1_list = []

    for c in range(num_classes):
        tp = np.sum((y_pred == c) & (y_true == c))
        fp = np.sum((y_pred == c) & (y_true != c))
        fn = np.sum((y_pred != c) & (y_true == c))

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1)

    return accuracy, np.mean(precision_list), np.mean(recall_list), np.mean(f1_list)

acc, prec, rec, f1 = compute_metrics(y_true_labels, predictions, C)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")

Accuracy:  0.0600
Precision: 0.0587
Recall:    0.0569
F1-score:  0.0552
